# How to define custom neural nets

`sbi` allows you to specify a specific density estimator for each of the implemented methods.
We support a variety of density estimators, e.g., mixtures of Gaussians, normalizing
flows, and diffusion models. Some of the density estimators are implemented as part of
`sbi`, for others we rely on other packages like
[`nflows`](https://github.com/bayesiains/nflows/) or [`zuko`](https://github.com/probabilists/zuko). 

For all options, check the API reference
[here](../api_reference/neural_nets.rst).

## Changing the type of density estimator

Pass a config object as `density_estimator` to `NPE` or `NLE`, for example `MAFConfig` for a masked autoregressive flow or `NSFConfig` for a neural spline flow.

Note that `MAFConfig` or `NSFConfig` correspond to `nflows` density
estimators. Those have proven to work well, but the `nflows` package is not maintained
anymore. To use more recent and actively maintained density estimators, we tentatively
recommend using `zuko`, e.g., `ZukoMAFConfig` or `ZukoNSFConfig`. 

In [ ]:
import torch

from sbi.inference import NPE, NRE
from sbi.utils import BoxUniform

In [ ]:
from sbi.neural_nets import ZukoMAFConfig

prior = BoxUniform(torch.zeros(2), torch.ones(2))
inference = NPE(prior=prior, density_estimator=ZukoMAFConfig())

In the case of `NRE`, the argument is called `classifier`:

In [ ]:
from sbi.neural_nets import ResNetClassifierConfig

inference = NRE(prior=prior, classifier=ResNetClassifierConfig())

## Changing hyperparameters of density estimators

Set hyperparameters when constructing the config.

Here we configure a Zuko neural spline flow with `60` hidden units and `3` transform layers:

In [ ]:
from sbi.neural_nets import ZukoNSFConfig

density_estimator = ZukoNSFConfig(hidden_features=60, num_transforms=3)
inference = NPE(prior=prior, density_estimator=density_estimator)

It is also possible to pass an `embedding_net` to a config to automatically
learn summary statistics from high-dimensional simulation outputs. You can find a more
detailed tutorial on this in [04_embedding_networks](https://sbi.readthedocs.io/en/latest/how_to_guide/04_embedding_networks.html).

See the [config guide](27_estimator_configs.ipynb) for available models, field validation, z-scoring, and migration examples.

## Building new density estimators from scratch

Finally, it is also possible to implement your own density estimator from scratch, e.g., including embedding nets to preprocess data, or to a density estimator architecture of your choice.

For this, the `density_estimator` argument needs to be a function that takes `theta` and `x` batches and returns an estimator. The trainer calls it after receiving simulations. Unlike this callable interface, `config.build` takes the modeled variable first: `build(theta, x)` for NPE and `build(x, theta)` for NLE.

The returned estimator must subclass `ConditionalDensityEstimator` from `sbi.neural_nets.estimators` and implement three methods:
    
- `log_prob(input, condition, **kwargs)`: Return the log probabilities of the inputs given a condition or multiple i.e. batched conditions.
- `loss(input, condition, **kwargs)`: Return the loss for training the density estimator.
- `sample(sample_shape, condition, **kwargs)`: Return samples from the density estimator.

See the [training interface tutorial](../advanced_tutorials/18_training_interface.ipynb) for custom training loops.